In [1]:
# Cell 1 — Setup
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy import io
import os, time, json
from tqdm import tqdm

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

RAW    = '/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/raw'
TEST   = '/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in'
STATS  = '/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/stats/feat_min_max.mat'

MONTHS       = ['APRIL_16', 'JULY_16', 'OCT_16', 'DEC_16']
ALL_FEATURES = ['cpm25', 'q2', 't2', 'u10', 'swdown', 'pblh', 'v10', 'psfc', 'rain']

TIME_IN  = 10
TIME_OUT = 16
S1, S2   = 140, 124

print(f"Features: {len(ALL_FEATURES)}")
print(f"Input steps: {TIME_IN}, Output steps: {TIME_OUT}")
print(f"Grid: {S1}x{S2}")

Device: cuda
Features: 9
Input steps: 10, Output steps: 16
Grid: 140x124


In [2]:
# Cell 2 — Robust Normalization
print("Computing robust normalization stats...")
stats = {}
for feat in ALL_FEATURES:
    all_data = []
    for month in MONTHS:
        data = np.load(f'{RAW}/{month}/{feat}.npy')
        all_data.append(data.flatten())
    all_data = np.concatenate(all_data)
    p25  = float(np.percentile(all_data, 25))
    p75  = float(np.percentile(all_data, 75))
    iqr  = p75 - p25 + 1e-8
    med  = float(np.median(all_data))
    stats[feat] = {'median': med, 'iqr': iqr}
    print(f"  {feat}: median={med:.3f}, iqr={iqr:.3f}")

np.save('/kaggle/working/norm_stats.npy', stats)
print("\nDone!")

Computing robust normalization stats...
  cpm25: median=16.128, iqr=32.616
  q2: median=0.013, iqr=0.014
  t2: median=298.506, iqr=18.805
  u10: median=1.287, iqr=4.662
  swdown: median=0.619, iqr=427.856
  pblh: median=638.636, iqr=587.888
  v10: median=-0.007, iqr=3.782
  psfc: median=97947.742, iqr=26628.320
  rain: median=0.000, iqr=0.000

Done!


In [3]:
# Cell 3 — Dataset
class PollutionDataset(Dataset):
    def __init__(self, split='train', val_frac=0.2):
        data = {f: [] for f in ALL_FEATURES}
        for month in MONTHS:
            for feat in ALL_FEATURES:
                arr = np.load(f'{RAW}/{month}/{feat}.npy')
                data[feat].append(arr)
        for feat in ALL_FEATURES:
            data[feat] = np.concatenate(data[feat], axis=0)

        T_total = data[ALL_FEATURES[0]].shape[0]
        horizon = TIME_IN + TIME_OUT
        indices = list(range(T_total - horizon + 1))
        np.random.seed(42)
        np.random.shuffle(indices)
        split_idx = int(len(indices) * (1 - val_frac))
        self.indices = indices[:split_idx] if split == 'train' else indices[split_idx:]
        self.data = data
        print(f"{split}: {len(self.indices)} samples")

    def _normalize(self, x, feat):
        med = stats[feat]['median']
        iqr = stats[feat]['iqr']
        return np.clip((x - med) / iqr, -10, 10).astype(np.float32)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        start = self.indices[idx]
        X = np.stack([
            self._normalize(self.data[f][start:start+TIME_IN], f)
            for f in ALL_FEATURES
        ], axis=-1)  # (TIME_IN, S1, S2, 9)
        Y = np.stack([
            # Keep target normalized for the loss function!
            self._normalize(self.data['cpm25'][start+TIME_IN:start+TIME_IN+TIME_OUT], 'cpm25')
        ], axis=-1).squeeze(-1) # (TIME_OUT, S1, S2) -> will permute in loss
        
        # Need Y to match model output shape: (S1, S2, TIME_OUT)
        Y = np.transpose(Y, (1, 2, 0)) 
        return torch.from_numpy(X), torch.from_numpy(Y)

train_ds     = PollutionDataset('train')
val_ds       = PollutionDataset('val')
# Added prefetch_factor and persistent_workers to stop CPU blocking
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,  num_workers=2, pin_memory=True, prefetch_factor=2, persistent_workers=True)
val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False, num_workers=2, pin_memory=True, prefetch_factor=2, persistent_workers=True)

train: 2325 samples
val: 582 samples


In [4]:
# Cell 4 — PredRNN++ Model
class SpatioTemporalLSTMCell(nn.Module):
    def __init__(self, in_ch, hidden_ch, kernel_size=3):
        super().__init__()
        p = kernel_size // 2
        self.hidden_ch = hidden_ch

        self.h_gates = nn.Conv2d(in_ch + hidden_ch, 4 * hidden_ch, kernel_size, padding=p)
        self.m_gates = nn.Conv2d(in_ch + hidden_ch, 3 * hidden_ch, kernel_size, padding=p)
        self.out     = nn.Conv2d(2 * hidden_ch, hidden_ch, 1)

    def forward(self, x, h, c, m):
        xh    = torch.cat([x, h], dim=1)
        gates = self.h_gates(xh)
        i, f, g, o = gates.chunk(4, dim=1)
        i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
        g     = torch.tanh(g)
        c_new = f * c + i * g

        xm      = torch.cat([x, m], dim=1)
        m_gates = self.m_gates(xm)
        i_m, f_m, g_m = m_gates.chunk(3, dim=1)
        i_m, f_m = torch.sigmoid(i_m), torch.sigmoid(f_m)
        g_m      = torch.tanh(g_m)
        m_new    = f_m * m + i_m * g_m

        h_new = o * torch.tanh(self.out(torch.cat([c_new, m_new], dim=1)))
        return h_new, c_new, m_new


class PredRNNPP(nn.Module):
    def __init__(self, in_ch=9, hidden_ch=64, num_layers=3,
                 time_in=10, time_out=16, s1=140, s2=124):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_ch  = hidden_ch
        self.time_out   = time_out
        self.s1, self.s2 = s1, s2

        self.input_proj = nn.Conv2d(in_ch, hidden_ch, 1)
        self.cells      = nn.ModuleList([
            SpatioTemporalLSTMCell(hidden_ch, hidden_ch)
            for _ in range(num_layers)
        ])
        self.output_head = nn.Sequential(
            nn.Conv2d(hidden_ch, hidden_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(hidden_ch, time_out, 1)
        )

    def _init_states(self, B):
        zeros = lambda: torch.zeros(B, self.hidden_ch, self.s1, self.s2, device=device)
        h = [zeros() for _ in range(self.num_layers)]
        c = [zeros() for _ in range(self.num_layers)]
        m = zeros()
        return h, c, m

    def forward(self, x):
        # x: (B, T, S1, S2, C)
        B, T, S1, S2, C = x.shape
        x = x.permute(0, 1, 4, 2, 3)  # (B, T, C, S1, S2)

        h, c, m = self._init_states(B)

        for t in range(T):
            inp = self.input_proj(x[:, t])
            for l, cell in enumerate(self.cells):
                h[l], c[l], m = cell(inp, h[l], c[l], m)
                inp = h[l]

        out = self.output_head(h[-1])   # (B, TIME_OUT, S1, S2)
        out = out.permute(0, 2, 3, 1)  # (B, S1, S2, TIME_OUT)
        return out


model = PredRNNPP(
    in_ch=9, hidden_ch=64, num_layers=3,
    time_in=TIME_IN, time_out=TIME_OUT, s1=S1, s2=S2
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"PredRNN++ parameters: {total_params:,}")

PredRNN++ parameters: 1,613,008


In [5]:
# Cell 5 — Episode-Aware Loss
class EpisodeAwareLoss(nn.Module):
    """
    MAE loss + extra penalty on episode (spike) grid points.
    Because target is already normalized via RobustScaler (median=0, IQR=1),
    an episode (target > median + 2*IQR) simply means normalized_target > 2.0.
    """
    def __init__(self, episode_weight=3.0, threshold=2.0):
        super().__init__()
        self.episode_weight = episode_weight
        self.threshold = threshold

    def forward(self, pred, target):
        # pred, target: (B, S1, S2, T)
        base_loss = F.l1_loss(pred, target)

        # Vectorized, no median/quantile calculation needed on the fly!
        episode_mask  = (target > self.threshold).float()
        
        # Avoid division by zero if no episodes in the batch
        mask_sum = episode_mask.sum()
        if mask_sum > 0:
            episode_loss  = (episode_mask * (pred - target).abs()).sum() / mask_sum
        else:
            episode_loss = 0.0

        return base_loss + self.episode_weight * episode_loss

criterion = EpisodeAwareLoss(episode_weight=5.0) # Bumped up the penalty slightly
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
print("Loss, optimizer, scheduler ready!")

Loss, optimizer, scheduler ready!


In [6]:
# Cell 6 — Training
EPOCHS    = 50
SAVE_PATH = '/kaggle/working/predrnn_best.pt'
LOG_PATH  = '/kaggle/working/predrnn_log.json'

best_val_loss = float('inf')
log = []

# Initialize the mixed precision scaler
scaler = torch.amp.GradScaler('cuda')

for ep in tqdm(range(EPOCHS)):
    # Train
    model.train()
    t0         = time.time()
    train_loss = 0.0
    
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        
        # Run forward pass in half precision
        with torch.amp.autocast('cuda'):
            pred = model(x)
            loss = criterion(pred, y)
            
        # Backward pass with scaled gradients
        scaler.scale(loss).backward()
        
        # Unscale before clipping to ensure proper gradient norms
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        # Step optimizer and update scaler
        scaler.step(optimizer)
        scaler.update()
        
        train_loss += loss.item()
    scheduler.step()

    # Validate
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            with torch.cuda.amp.autocast():
                val_loss += criterion(model(x), y).item()

    train_loss /= len(train_loader)
    val_loss   /= len(val_loader)
    elapsed     = time.time() - t0

    log.append({'epoch': ep, 'train_loss': train_loss, 'val_loss': val_loss})
    print(f"Epoch {ep:3d} | {elapsed:.1f}s | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({'epoch': ep, 'model_state_dict': model.state_dict()}, SAVE_PATH)
        print(f"  *** Best model saved! Val Loss: {val_loss:.4f} ***")

    with open(LOG_PATH, 'w') as f:
        json.dump(log, f)

print("Training complete!")

  0%|          | 0/50 [00:00<?, ?it/s]/tmp/ipykernel_55/1081881786.py:47: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
  2%|▏         | 1/50 [07:55<6:28:38, 475.88s/it]

Epoch   0 | 475.9s | Train: 5.4495 | Val: 4.6387
  *** Best model saved! Val Loss: 4.6387 ***


  4%|▍         | 2/50 [15:55<6:22:33, 478.21s/it]

Epoch   1 | 479.8s | Train: 4.4431 | Val: 4.4324
  *** Best model saved! Val Loss: 4.4324 ***


  6%|▌         | 3/50 [23:53<6:14:29, 478.08s/it]

...

Epoch  47 | 478.6s | Train: 1.9021 | Val: 2.1710
  *** Best model saved! Val Loss: 2.1710 ***


 98%|█████████▊| 49/50 [6:28:51<07:58, 478.53s/it]

Epoch  48 | 480.0s | Train: 1.9112 | Val: 2.1706
  *** Best model saved! Val Loss: 2.1706 ***


100%|██████████| 50/50 [6:36:52<00:00, 476.25s/it]

Epoch  49 | 480.8s | Train: 1.9114 | Val: 2.1704
  *** Best model saved! Val Loss: 2.1704 ***
Training complete!


In [7]:
# Cell 7 — Inference
# Load best model
ckpt = torch.load(SAVE_PATH, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Loaded best model from epoch {ckpt['epoch']}")

class TestDataset(Dataset):
    def __init__(self):
        self.arrs = {
            f: np.load(f'{TEST}/{f}.npy')  # (218, 10, 140, 124)
            for f in ALL_FEATURES
        }
        self.N = 218

    def __len__(self): return self.N

    def _normalize(self, x, feat):
        med = stats[feat]['median']
        iqr = stats[feat]['iqr']
        return np.clip((x - med) / iqr, -10, 10).astype(np.float32)

    def __getitem__(self, idx):
        X = np.stack([
            self._normalize(self.arrs[f][idx], f)  # (10, 140, 124)
            for f in ALL_FEATURES
        ], axis=-1)  # (10, 140, 124, 9)
        return torch.from_numpy(X)

test_ds     = TestDataset()
test_loader = DataLoader(test_ds, batch_size=4, shuffle=False, num_workers=0)

predictions = np.zeros((218, S1, S2, TIME_OUT), dtype=np.float32)

with torch.no_grad():
    idx = 0
    for x in tqdm(test_loader):
        x    = x.to(device)
        # Apply AMP to inference too for speed!
        with torch.amp.autocast('cuda'):
            pred = model(x)        # (B, S1, S2, TIME_OUT)
        
        B    = x.size(0)
        predictions[idx:idx+B] = pred.cpu().numpy()
        idx += B

# ==========================================
# THIS IS THE DENORMALIZATION BLOCK
# ==========================================
# We multiply by IQR and add the median to get back to raw µg/m³
cpm25_med   = stats['cpm25']['median']
cpm25_iqr   = stats['cpm25']['iqr']
predictions = (predictions * cpm25_iqr) + cpm25_med
predictions = np.clip(predictions, 0, None)  # PM2.5 can't be negative

np.save('/kaggle/working/preds.npy', predictions)
print("Saved! Shape:", predictions.shape)
print("Expected:     (218, 140, 124, 16)")
print(f"PM2.5 range: [{predictions.min():.2f}, {predictions.max():.2f}] µg/m³")

Loaded best model from epoch 49


100%|██████████| 55/55 [00:17<00:00,  3.11it/s]


Saved! Shape: (218, 140, 124, 16)
Expected:     (218, 140, 124, 16)
PM2.5 range: [0.00, 395.54] µg/m³
